New mosaics (and new bands), and updated data version numbers

In [2]:
import ee
import pandas as pd

In [3]:
ee.Initialize(project='mapbiomas-india')

*** Earth Engine *** Share your feedback by taking our Annual Developer Satisfaction Survey: https://google.qualtrics.com/jfe/form/SV_9oS0DRcPvElRMNw?source=python


1. Get total samples per year for each region

In [4]:
regions = ['HIM', 'IGP', 'DES', 'PEN', 'NEI', 'WGR', 'ICR']
region_id_map = {'HIM': 1, 'IGP': 2, 'DES': 3, 'PEN': 4, 'NEI': 5, 'WGR': 6, 'ICR': 7}

# Define regional sample versions 
# (Update these values with your actual version strings)
samples_version_map = {
    'HIM': '7', 
    'IGP': '7', 
    'DES': '8', 
    # 'PEN': '5', 
    # 'NEI': '5', 
    'WGR': '5', 
    'ICR': '7'
}

In [5]:
def get_region_sample_counts(
    year: int,
    samples_folder: str = 'projects/mapbiomas-india/assets/LAND-COVER/COLLECTION-1/SAMPLES/',
    region_id_map: dict = None,
    samples_version_map: dict = None,
    territory_name:str='INDIA'
) -> pd.DataFrame:
    """
    Computes total sample counts per region for a specified year
    and returns a formatted DataFrame.
    """
    if region_id_map is None:
        region_id_map = {'HIM': 1, 'IGP': 2, 'DES': 3, 'PEN': 4, 'NEI': 5, 'WGR': 6, 'ICR': 7}
    
    if samples_version_map is None:
        samples_version_map = {'HIM': '7', 'IGP': '7', 'DES': '8', 'WGR': '5', 'ICR': '7'}

    records = []

    for region, version in samples_version_map.items():
        region_id = region_id_map.get(region)
        if region_id is None:
            continue

        asset_name = f"{territory_name}_training_samples_region{region_id}_v{version}_merged"
        asset_path = f"{samples_folder.rstrip('/')}/{asset_name}"

        try:
            fc = ee.FeatureCollection(asset_path).filter(ee.Filter.eq('year', year))
            count = fc.size().getInfo()
        except ee.EEException as e:
            count = None

        records.append({
            'Region': region,
            'Region ID': region_id,
            'Version': version,
            'Year': year,
            'Total Samples': count
        })

    df = pd.DataFrame(records)
    return df

In [5]:
get_region_sample_counts(
    year = 2024,
    region_id_map = region_id_map,
    samples_version_map = samples_version_map
)

,Region,Region ID,Version,Year,Total Samples
0,HIM,1,7,2024,48920
1,IGP,2,7,2024,81813
2,DES,3,8,2024,117387
3,WGR,6,5,2024,63193
4,ICR,7,7,2024,94052


In [6]:
get_region_sample_counts(
    year = 2015,
    region_id_map = region_id_map,
    samples_version_map = samples_version_map
)

,Region,Region ID,Version,Year,Total Samples
0,HIM,1,7,2015,48657
1,IGP,2,7,2015,82587
2,DES,3,8,2015,117387
3,WGR,6,5,2015,63193
4,ICR,7,7,2015,94052


2. Setup optuna study for one region, for specific years

In [1]:
YEAR = [2015]

In [6]:
def export_samples_to_drive(
    regions: list,
    years: list,
    drive_folder: str = "GEE_EXPORTS",
    samples_folder: str = "projects/mapbiomas-india/assets/LAND-COVER/COLLECTION-1/SAMPLES/",
    region_id_map: dict = None,
    sample_version_map: dict = None,
    territory_name: str = 'INDIA'
):
    if region_id_map is None:
        region_id_map = {'HIM': 1, 'IGP': 2, 'DES': 3, 'PEN': 4, 'NEI': 5, 'WGR': 6, 'ICR': 7}
    if sample_version_map is None:
        sample_version_map = {'HIM': '7', 'IGP': '7', 'DES': '8', 'WGR': '5', 'ICR': '7'}

    for region in regions:
        if region not in sample_version_map:
            continue
        region_id = region_id_map[region]
        version = sample_version_map[region]
        asset_path = f"{samples_folder.rstrip('/')}/{territory_name}_training_samples_region{region_id}_v{version}_merged"

        for year in years:
            # Filter by year and explicitly discard geometry payload during table export
            fc = (ee.FeatureCollection(asset_path)
                  .filter(ee.Filter.eq('year', year))
                  .select(propertySelectors=['.*'], retainGeometry=False))

            description = f"{region}_{year}_v{version}"
            task = ee.batch.Export.table.toDrive(
                collection=fc,
                description=description,
                folder=drive_folder,
                fileNamePrefix=description,
                fileFormat='CSV'
            )
            task.start()
            print(f"Export task queued: {description} (Task ID: {task.id})")

In [8]:
target_regions = ['DES']
target_years = [1995,2005]

export_samples_to_drive(
    regions=target_regions,
    years=target_years,
    drive_folder="GEE_EXPORTS/IOLN_new_mosaics",
)

Export task queued: DES_1995_v8 (Task ID: ZKAMHFXBHNWB7YCWEJY7T45M)
Export task queued: DES_2005_v8 (Task ID: FOOKDDY6VZQM3G53XQN7ESBV)
